# RidgeRegression (gradient descent) and Exercise 8: RidgeRegressionLeastSquares on cpu.csv

In [1]:
import sys
from pathlib import Path

import numpy as np

# find the repository root (the folder that contains "datasets"), wherever the notebook is
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'datasets').is_dir())
for path in (ROOT / 'src', ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

DATA = ROOT / 'datasets'

from si.io.csv_file import read_csv
from si.model_selection.split import train_test_split
from si.models.linear_regression import RidgeRegression
from si.models.ridge_regression_least_squares import RidgeRegressionLeastSquares

In [2]:
cpu = read_csv(str(DATA / 'cpu' / 'cpu.csv'), features=True, label=True)
train, test = train_test_split(cpu, test_size=0.2, random_state=42)

## Gradient descent
Which score (MSE) and cost do you get?

In [3]:
gd = RidgeRegression(l2_penalty=1.0, alpha=0.01, max_iter=2000, patience=5, scale=True)
gd.fit(train)
print('iterations:', len(gd.cost_history))
print('score (MSE) test:', round(gd.score(test), 2))
print('cost test:', round(gd.cost(test), 2))

iterations: 2000
score (MSE) test: 3616.73
cost test: 1908.21


### Cost history
The cost drops quickly at first and then slowly, as the gradient gets smaller.

In [4]:
costs = list(gd.cost_history.values())
for i in [0, 1, 10, 100, 500, len(costs) - 1]:
    print(f'iteration {i:>5}: cost {costs[i]:.2f}')

iteration     0: cost 14608.40
iteration     1: cost 14053.98
iteration    10: cost 10238.21
iteration   100: cost 2806.09
iteration   500: cost 1884.87
iteration  1999: cost 1877.83


### Effect of the learning rate

In [5]:
for alpha in [0.0001, 0.001, 0.01, 0.1]:
    m = RidgeRegression(alpha=alpha, max_iter=2000).fit(train)
    print(f'alpha={alpha:<7} iterations={len(m.cost_history):>5}  test MSE={m.score(test):.2f}')

alpha=0.0001  iterations= 2000  test MSE=27780.28
alpha=0.001   iterations= 2000  test MSE=3850.28
alpha=0.01    iterations= 2000  test MSE=3616.73
alpha=0.1     iterations=  875  test MSE=3638.06


## Least squares (analytical solution)
One linear system instead of thousands of iterations. Gradient descent converges to the same parameters.

In [6]:
ls = RidgeRegressionLeastSquares(l2_penalty=1.0, scale=True).fit(train)
print('score (MSE) test:', round(ls.score(test), 2))
gd_long = RidgeRegression(l2_penalty=1.0, alpha=0.1, max_iter=20000).fit(train)
print('theta_zero  GD:', round(gd_long.theta_zero, 4), '| LS:', round(ls.theta_zero, 4))
print('theta GD:', gd_long.theta.round(3))
print('theta LS:', ls.theta.round(3))

score (MSE) test: 3638.06
theta_zero  GD: 97.1845 | LS: 97.1845
theta GD: [ 9.936 30.015 70.491 29.1   -1.352 37.198]
theta LS: [ 9.936 30.015 70.491 29.1   -1.352 37.198]
